In [1]:
import pandas as pd

df = pd.read_parquet("data/bpi2017.parquet")
case, act, ts = "case:concept:name", "concept:name", "time:timestamp"
print("Rows loaded:", len(df))

# 1. Exact duplicate rows
print("\n1. Exact duplicate rows:", df.duplicated().sum())

# 2. Duplicate event IDs
print("2. Duplicate EventIDs:", df["EventID"].duplicated().sum())

# 3. Requested amount per application
ra = df.groupby(case)["case:RequestedAmount"].first()
print("3. Requested amount -> min:", ra.min(), "| max:", ra.max(),
      "| zero:", (ra == 0).sum(), "| negative:", (ra < 0).sum())

# 4. First step of each application
first = df.sort_values([case, ts]).groupby(case)[act].first()
print("\n4. First step per application:")
print(first.value_counts().to_string())

# 5. Rows with the same time in the same application
print("\n5. Rows sharing exact time within an app:",
      df.duplicated([case, ts], keep=False).sum())

# 6. Lifecycle values
print("\n6. Lifecycle values:")
print(df["lifecycle:transition"].value_counts().to_string())

# 7. Staff / resources
print("\n7. Number of resources:", df["org:resource"].nunique())
print(df["org:resource"].value_counts().head(3).to_string())

# 8. Ending recorded twice
endings = ["A_Pending", "A_Cancelled", "A_Denied"]
e = df[df[act].isin(endings)].groupby(case).size()
print("\n8. Apps with ending recorded twice:", (e > 1).sum())

# 9. Open apps: when did they last move?
open_last = df[~df[case].isin(e.index)].groupby(case)[ts].max()
print("9. Open apps:", len(open_last),
      "| earliest last event:", open_last.min(),
      "| latest last event:", open_last.max())

Rows loaded: 1202267

1. Exact duplicate rows: 0
2. Duplicate EventIDs: 0
3. Requested amount -> min: 0.0 | max: 450000.0 | zero: 3080 | negative: 0

4. First step per application:
concept:name
A_Create Application    31509

5. Rows sharing exact time within an app: 0

6. Lifecycle values:
lifecycle:transition
complete     475306
suspend      215402
schedule     149104
start        128227
resume       127160
ate_abort     85224
withdraw      21844

7. Number of resources: 149
org:resource
User_1    148404
User_3     26342
User_5     22900

8. Apps with ending recorded twice: 1
9. Open apps: 98 | earliest last event: 2017-01-02 08:55:26.535000+00:00 | latest last event: 2017-02-01 14:11:03.499000+00:00


In [2]:
# A. OfferID: which offer steps have it?
offer = df[df["EventOrigin"] == "Offer"]
print("A. Share of rows with OfferID, per offer step:")
print(offer.groupby(act)["OfferID"].apply(lambda s: round(s.notna().mean(), 3)).to_string())

# B. User_1: what does it do?
print("\nB. Top steps done by User_1:")
print(df[df["org:resource"] == "User_1"][act].value_counts().head(5).to_string())

# C. Zero-amount apps: what loan goals?
zero_apps = ra[ra == 0].index
goals = df[df[case].isin(zero_apps)].groupby(case)["case:LoanGoal"].first()
print("\nC. Loan goals of zero-amount apps:")
print(goals.value_counts().head(5).to_string())

A. Share of rows with OfferID, per offer step:
concept:name
O_Accepted                  1.0
O_Cancelled                 1.0
O_Create Offer              0.0
O_Created                   1.0
O_Refused                   1.0
O_Returned                  1.0
O_Sent (mail and online)    1.0
O_Sent (online only)        1.0

B. Top steps done by User_1:
concept:name
W_Handle leads            37453
A_Create Application      20423
A_Submitted               20423
W_Complete application    17120
A_Concept                 16997

C. Loan goals of zero-amount apps:
case:LoanGoal
Unknown                   2055
Home improvement           309
Car                        284
Existing loan takeover     185
Other, see explanation     121


## Data quality notes

Before looking for bottlenecks, I wanted to make sure the data could be trusted. Overall it's in good shape. There are no duplicate rows, no repeated event IDs, and every one of the 31,509 applications starts with "A_Create Application", which is what you'd expect.

A few things needed a closer look though.

About 3,080 applications have a requested amount of 0. When I checked, two out of three of them also had "Unknown" as the loan goal, so my guess is the customer just didn't fill it in. I'm keeping these applications, but I won't use them when I calculate average loan amounts.

The offer columns (amount, monthly cost, credit score and so on) are empty for 96% of rows. That looked bad at first, but it turns out they're only filled in when an offer is created. Every other step simply doesn't carry that information.

Work steps are logged in pieces. One task can show up as "start", "suspend", "resume" and "complete", so counting rows would overstate the workload. I'll count a task only when it's completed.

User_1 shows up on 12% of all events and created 20,423 applications. That's far too much for one person, so it's almost certainly the online system. I'll leave it out when I look at staff workload. It also suggests there are two ways in: customers applying online, and staff entering applications themselves (11,086 of them).

98 applications never reach an outcome. The data stops on 1 February 2017, so they were most likely still in progress. I'll leave them out when measuring how long applications take. The same problem affects applications from late 2016, since they had less time to finish. I'll come back to that in the bottleneck analysis.

One application has its final outcome logged twice. Small slip, I'll just count it once.